In [2]:
import pandas as pd
import numpy as np
import pickle

import warnings

from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

In [3]:
warnings.filterwarnings("ignore")

In [4]:
df = pd.read_csv("../../data/pre_data.csv")

In [5]:
df.head()

,Tên_mặt_hàng,Thị_trường,Loại_giá,Nguồn,Ngày,Giá
0,Gạo NL 25% tấm,Kiên Giang,Thương lái thu mua,CTV địa phương,2025-05-16,8460.0
1,Gạo XK 5% tấm,Kiên Giang,Thương lái thu mua,CTV địa phương,2025-05-16,10070.0
2,Gạo XK 5% tấm,Tiền Giang,Thương lái thu mua,CTV địa phương,2025-05-14,15300.0
3,ST24,Hậu Giang,Thương lái thu mua,CTV địa phương,2025-05-13,10650.0
4,Gạo NL 25% tấm,Kiên Giang,Thương lái thu mua,CTV địa phương,2025-05-09,8520.0


In [6]:
for col in ["Tên_mặt_hàng", "Thị_trường", "Loại_giá", "Nguồn"]:
    lbl_encoder = LabelEncoder()
    df[col] = lbl_encoder.fit_transform(df[[col]])

    with open(f"../lbl_scaler/{col}.pkl", "wb") as file:
        pickle.dump(lbl_encoder, file)

In [7]:
for col in ["Tên_mặt_hàng", "Thị_trường", "Loại_giá", "Nguồn"]:
    scaler = MinMaxScaler()  
    df[col] = scaler.fit_transform(df[[col]])

    with open(f"../mm_scaler/{col}.pkl", "wb") as file:
        pickle.dump(scaler, file)

In [14]:
def get_mm_dict():
    cols = ["Tên_mặt_hàng", "Thị_trường", "Loại_giá", "Nguồn"]
    result = dict()

    for col in cols:
        with open(f"../mm_scaler/{col}.pkl", "rb") as file:
            mm_scaler = pickle.load(file)
            result.update({col: mm_scaler})
    
    return result

def get_lbl_dict():
    cols = ["Tên_mặt_hàng", "Thị_trường", "Loại_giá", "Nguồn"]
    result = dict()

    for col in cols:
        with open(f"../lbl_scaler/{col}.pkl", "rb") as file:
            mm_scaler = pickle.load(file)
            result.update({col: mm_scaler})
    
    return result

In [13]:
mm_dict = get_mm_dict()
mm_dict

{'Tên_mặt_hàng': MinMaxScaler(),
 'Thị_trường': MinMaxScaler(),
 'Loại_giá': MinMaxScaler(),
 'Nguồn': MinMaxScaler()}

In [15]:
lbl_dict = get_lbl_dict()
lbl_dict

{'Tên_mặt_hàng': LabelEncoder(),
 'Thị_trường': LabelEncoder(),
 'Loại_giá': LabelEncoder(),
 'Nguồn': LabelEncoder()}

In [16]:
groups = df.groupby(["Tên_mặt_hàng", "Thị_trường", "Loại_giá", "Nguồn"])
results = []

for keys, group_df in groups:
    group_df = group_df.sort_values("Ngày")
    
    if len(group_df) < 10:
        continue  # Skip if not enough data

    y = group_df["Giá"]
    exog = group_df[["Tên_mặt_hàng", "Thị_trường", "Loại_giá", "Nguồn"]]

    try:
        model = SARIMAX(y, exog=exog, order=(1,1,1), seasonal_order=(0,0,0,0))
        model_fit = model.fit(disp=False)
        with open("./model.pkl", "wb") as file:
            pickle.dump(model_fit, file)
        
        # Forecast using the last row of exog
        forecast = model_fit.forecast(steps=1, exog=exog.tail(1))
        results.append((keys, forecast.iloc[0]))

    except Exception as e:
        print(f"Skip group {keys} due to error: {e}")
